In [1]:
import math
import pandas as pd
import random

# Part1  Utility Functions (From Scratch)

In [2]:

def compute_mean(values):
    total = 0.0
    n = len(values)
    for v in values:
        total += v
    return total / n


def compute_variance(values):
    mu = compute_mean(values)
    total = 0.0
    n = len(values)
    for v in values:
        total += (v - mu) ** 2
    return total / n


def compute_accuracy(y_true, y_pred):
    correct = 0
    total = len(y_true)
    for i in range(total):
        if y_true[i] == y_pred[i]:
            correct += 1
    return correct / total

# Part2 Gaussian Naive Bayes (Abalone)

### Load Dataset

In [3]:
df = pd.read_csv("abalone.csv")
df

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9
4175,F,0.625,0.485,0.150,1.0945,0.5310,0.2610,0.2960,10


### Convert Rings to age class

In [4]:
def rings_to_class(r):
    if r <= 8:
        return "Young"
    elif r <= 11:
        return "Adult"
    else:
        return "Old"

df["AgeClass"] = df["Rings"].apply(rings_to_class)

### Select Features

In [5]:
features = [
    "Length", "Diameter", "Height",
    "Whole weight", "Shucked weight",
    "Viscera weight", "Shell weight"
]

X = df[features].values.tolist()
y = df["AgeClass"].tolist()

### Split dataset (80% training / 20% testing)

In [6]:
data = list(zip(X, y))
random.shuffle(data)

split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

X_train = [d[0] for d in train_data]
y_train = [d[1] for d in train_data]
X_test  = [d[0] for d in test_data]
y_test  = [d[1] for d in test_data]

### Compute class priors


In [7]:
def compute_priors(labels):
    priors = {}
    total = len(labels)
    for c in labels:
        priors[c] = priors.get(c, 0) + 1
    for c in priors:
        priors[c] /= total
    return priors

priors = compute_priors(y_train)

### Compute mean and variance for each feature per class

In [8]:
def summarize_by_class(X, y):
    separated = {}
    for i in range(len(y)):
        label = y[i]
        features = X[i]
        if label not in separated:
            separated[label] = []
        separated[label].append(features)

    summaries = {}
    for label, rows in separated.items():
        summaries[label] = []
        num_features = len(rows[0])
        for j in range(num_features):
            col = [row[j] for row in rows]
            mu = compute_mean(col)
            var = compute_variance(col)
            summaries[label].append((mu, var))
    return summaries

summaries = summarize_by_class(X_train, y_train)

### Gaussian probability density function

In [9]:
def gaussian_pdf(x, mean, var):
    eps = 1e-9
    coeff = 1.0 / math.sqrt(2 * math.pi * var + eps)
    exponent = math.exp(-((x - mean) ** 2) / (2 * var + eps))
    return coeff * exponent

In [10]:
def predict_prob(sample, priors, summaries):
    probs = {}
    for label in priors:
        probs[label] = math.log(priors[label])
        for i in range(len(sample)):
            mu, var = summaries[label][i]
            p = gaussian_pdf(sample[i], mu, var)
            probs[label] += math.log(p + 1e-9)
    return probs

### Predict test samples

In [11]:
def predict(sample, priors, summaries):
    probs = predict_prob(sample, priors, summaries)
    return max(probs, key=probs.get)

In [12]:
y_pred = []
for sample in X_test:
    label = predict(sample, priors, summaries)
    y_pred.append(label)

### Compute accuracy

In [13]:
acc = compute_accuracy(y_test, y_pred)
print("Accuracy:", acc)

Accuracy: 0.5717703349282297
